# Linear player model

- **Question:** What does a regularized linear model learn beyond transparent player baselines?
- **Data used:** Canonical cutoff-safe `player_season_features` and `player_season_targets`, plus persisted Phase 4 model metadata and predictions when available.
- **Unit of observation:** One non-rookie player and prediction season, routed by QB, RB, WR, or TE.
- **Target:** Next-season fantasy points per active game, games active, or total fantasy points; the diagnostic example focuses on WR points per active game.
- **Feature cutoff:** A prediction-season row uses only information available before that season begins.
- **Validation strategy:** Expanding 2020-2024 validation folds, chronological tuning inside each training period, then an untouched 2025 test.
- **Interpretation caveat:** Coefficients and reference substitutions describe associations in this fitted dataset, not causal effects. Historical rookie rows are not available for learned-model validation.

## Local prerequisites

Run the Phase 3 feature and baseline commands, then the Phase 4 player-model command from the repository root. This notebook reads the local warehouse and report without modifying either. When Phase 4 outputs are absent, the cells show the contract and print the missing prerequisite instead of inventing results.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the repository root.")


def table_exists(connection: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
        [table_name],
    ).fetchone()
    return bool(row and row[0])


PROJECT_ROOT = find_project_root()
WAREHOUSE_PATH = PROJECT_ROOT / "data" / "warehouse" / "fantasy_football.duckdb"
REPORT_PATH = PROJECT_ROOT / "docs" / "PHASE_4_MODEL_EVALUATION.json"
print({"warehouse_ready": WAREHOUSE_PATH.is_file(), "report_ready": REPORT_PATH.is_file()})

## Prove the time order first

The fold labels are prediction seasons. Every training season must be strictly earlier than the row being evaluated; the final fold is test-only.

In [ ]:
from fantasy_draft_ai.models.evaluation.splits import expanding_season_splits

folds = expanding_season_splits(
    range(2016, 2026),
    first_evaluation_season=2020,
    last_evaluation_season=2025,
    min_training_seasons=2,
)
fold_audit = pd.DataFrame(
    [
        {
            "evaluation_season": fold.evaluation_season,
            "fold_label": fold.label,
            "training_seasons": ", ".join(map(str, fold.training_seasons)),
            "training_max_season": max(fold.training_seasons),
            "passes_order_check": max(fold.training_seasons) < fold.evaluation_season,
        }
        for fold in folds
    ]
)
assert fold_audit["passes_order_check"].all()
fold_audit

## Inspect the locked Ridge contract

All learned preprocessing lives inside the pipeline. The allowlist prevents a newly added payload field, baseline output, or target-derived value from silently becoming a predictor.

In [ ]:
from fantasy_draft_ai.models.player_projection.config import RIDGE, PlayerModelConfig
from fantasy_draft_ai.models.player_projection.pipelines import candidate_parameters

config = PlayerModelConfig()
contract = pd.DataFrame(
    {
        "feature": (*config.numeric_features, *config.categorical_features),
        "kind": (
            *(["numeric"] * len(config.numeric_features)),
            *(["categorical"] * len(config.categorical_features)),
        ),
    }
)
print("Ridge candidates:", candidate_parameters(RIDGE, config))
print("Feature-contract fingerprint:", config.feature_contract_fingerprint())
contract

In [ ]:
ridge_models = pd.DataFrame()
ridge_predictions = pd.DataFrame()
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if table_exists(connection, "player_projection_models"):
            ridge_models = connection.execute(
                """
                SELECT run_id, model_id, position, target_name, training_seasons,
                       training_rows, hyperparameters, uncertainty_method, model_card_path
                FROM player_projection_models
                WHERE model_family = 'ridge'
                ORDER BY position, target_name
                """
            ).df()
        if table_exists(connection, "player_projection_predictions"):
            ridge_predictions = connection.execute(
                """
                SELECT player_id, prediction_season, position, target_name, fold_label,
                       predicted_value, actual_value, training_max_season
                FROM player_projection_predictions
                WHERE model_family = 'ridge' AND actual_value IS NOT NULL
                ORDER BY prediction_season, position, target_name, player_id
                """
            ).df()

if ridge_models.empty:
    print("No persisted Ridge models yet. Run the Phase 4 player-model training command.")
else:
    display(ridge_models)

In [ ]:
if ridge_predictions.empty:
    print("No evaluable Ridge predictions are stored yet.")
else:
    assert (ridge_predictions["training_max_season"] < ridge_predictions["prediction_season"]).all()
    ridge_metrics = (
        ridge_predictions.assign(
            absolute_error=lambda frame: (frame["predicted_value"] - frame["actual_value"]).abs()
        )
        .groupby(["fold_label", "prediction_season", "position", "target_name"], dropna=False)
        .agg(rows=("absolute_error", "size"), mae=("absolute_error", "mean"))
        .reset_index()
    )
    display(ridge_metrics)

## A statsmodels coefficient diagnostic

The next cell fits a deliberately small OLS diagnostic on real cutoff-safe WR rows when they exist. It is not the production Ridge pipeline, does not select a champion, and does not write an artifact. Its purpose is to practice coefficient direction, confidence intervals, and residual thinking with familiar columns.

In [ ]:
import statsmodels.api as sm

diagnostic_rows: list[dict[str, float | int | str]] = []
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if table_exists(connection, "player_season_features") and table_exists(
            connection, "player_season_targets"
        ):
            raw = connection.execute(
                """
                SELECT f.player_id, f.prediction_season, f.feature_payload::VARCHAR AS features,
                       t.target_payload::VARCHAR AS targets
                FROM player_season_features AS f
                JOIN player_season_targets AS t USING (player_id, prediction_season)
                WHERE f.position = 'WR' AND f.prediction_season <= 2024
                ORDER BY f.prediction_season, f.player_id
                """
            ).df()
            for row in raw.itertuples(index=False):
                features = json.loads(row.features)
                targets = json.loads(row.targets)
                target = targets.get("fantasy_points_per_game")
                if features.get("is_rookie") or target is None:
                    continue
                diagnostic_rows.append(
                    {
                        "player_id": row.player_id,
                        "prediction_season": row.prediction_season,
                        "age_at_cutoff": features.get("age_at_cutoff"),
                        "weighted_3yr_fantasy_points_per_game": features.get(
                            "weighted_3yr_fantasy_points_per_game"
                        ),
                        "weighted_3yr_targets_per_game": features.get(
                            "weighted_3yr_targets_per_game"
                        ),
                        "fantasy_points_per_game": target,
                    }
                )

diagnostic = pd.DataFrame(diagnostic_rows)
predictors = [
    "age_at_cutoff",
    "weighted_3yr_fantasy_points_per_game",
    "weighted_3yr_targets_per_game",
]
if diagnostic.empty:
    complete_training = pd.DataFrame(columns=[*predictors, "fantasy_points_per_game"])
else:
    complete_training = (
        diagnostic.loc[diagnostic["prediction_season"].lt(2024)]
        .dropna(subset=[*predictors, "fantasy_points_per_game"])
        .copy()
    )
if len(complete_training) < 20:
    print("Not enough complete pre-2024 WR rows for the OLS diagnostic.")
else:
    design = sm.add_constant(complete_training[predictors], has_constant="add")
    ols = sm.OLS(complete_training["fantasy_points_per_game"], design).fit()
    coefficient_table = pd.concat(
        [
            ols.params.rename("coefficient"),
            ols.conf_int().rename(columns={0: "ci_lower", 1: "ci_upper"}),
        ],
        axis=1,
    )
    display(coefficient_table)

## What to take away

Ridge earns its place through out-of-time comparison, not coefficient neatness. Inspect one stored model card after training, identify the selected penalty, and explain one coefficient with the phrase *is associated with*. Then compare the validation result with the best baseline without using the 2025 test to change the winner.